<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/Bayesian_Inference_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## 1. Visualizing the Mechanics

In the 2PL model, the difficulty parameter $b_i$ determines the horizontal location of the Item Response Function (IRF), while the discrimination parameter $a_i$ determines its steepness. 

When you increase $b_i$, the curve shifts horizontally to the **right**. This means a user requires a higher latent ability $\theta$ to have a 50% chance of answering the item correctly, indicating a more difficult question.

In [1]:
import numpy as np
import plotly.graph_objects as go

# Define theta range
theta = np.linspace(-4, 4, 400)

# 2PL function
def p_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()

# Curve 1, 2, 3: Same discrimination (a=1.5), different difficulties
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, -1), name="a=1.5, b=-1 (Easy)"))
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, 0), name="a=1.5, b=0 (Medium)"))
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 1.5, 1), name="a=1.5, b=1 (Hard)"))

# Curve 4: Different discrimination (a=0.5), medium difficulty
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 0.5, 0), 
                         name="a=0.5, b=0 (Low Discrimination)", 
                         line=dict(dash='dash')))

fig.update_layout(title="2PL Item Response Functions",
                  xaxis_title="Latent Ability (theta)",
                  yaxis_title="Probability of Correct Response P(Y=1)",
                  template="plotly_white")
fig.show()

 

## 2. Sequential Likelihood Contribution

Because the item responses are assumed to be conditionally independent given the latent ability $\theta$, the likelihood of a single response follows a Bernoulli distribution. 

The likelihood contribution of a single new response $y_k$ at step $k$ is:

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

Substituting the 2PL model formula, this becomes:

$$L(y_k \mid \theta) = \left( \frac{1}{1+e^{-a_k(\theta-b_k)}} \right)^{y_k} \left( \frac{e^{-a_k(\theta-b_k)}}{1+e^{-a_k(\theta-b_k)}} \right)^{1-y_k}$$

The joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods up to step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} L(y_j \mid \theta)$$

## 3. Mathematical Formulation of the Running Update

By Bayes' theorem, the posterior density at step $k$ is proportional to the product of the likelihood of the new observation and the prior density (which is the posterior from step $k-1$).

The recursive relationship is:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

To formulate the exact probability density function, normalize by dividing by the marginal likelihood:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})}{\int_{-\infty}^{\infty} L(y_k \mid u) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(u \mid \mathbf{y}^{(k-1)}) \, du}$$

## 4. Dynamic Shifting

If a user answers a highly difficult item correctly ($y_k = 1$ and $b_k$ is large), the likelihood function $L(y_k=1 \mid \theta)$ evaluates near zero for low $\theta$ values and approaches 1 for $\theta$ values greater than $b_k$. 

Multiplying the previous posterior density by this right-leaning likelihood function heavily penalizes the probability mass at the lower end of the ability scale and preserves it at the higher end. Consequently, the peak of the new running posterior density distribution shifts mathematically to the right toward a higher ability estimate.

## 5. Tracking Certainty and Sharpness

The discrimination parameter $a_k$ determines how steeply the response probability transitions from 0 to 1. 

*   **When $a_k$ is very large:** The likelihood function acts almost like a step function. It drastically slashes the probability mass on one side of $b_k$ while preserving the other side. This aggressive filtering rapidly reduces the variance of the posterior distribution, resulting in a sharper density and increasing certainty about the user's ability.
*   **When $a_k$ is very small:** The likelihood function is flat and uninformative. Multiplying the previous posterior by a nearly flat curve minimally affects its shape. The variance remains largely unaffected, indicating little certainty was gained from that interaction.

## 6. Numerical Implementation of a Running Grid

Because the exact posterior lacks a closed-form solution (it is not conjugate to the standard normal prior), it must be approximated numerically over a discrete grid.

1.  **Define the Grid:** Create a linearly spaced array of $\theta$ values (e.g., from -4 to 4 with 1000 points). Let the distance between points be $\Delta\theta$.
2.  **Initialize Prior:** Calculate the standard normal density for each point on the grid. This array is the starting posterior $k=0$.
3.  **Update Loop:** When a new response $y_k$ is observed:
    *   Calculate the likelihood array for every point on the grid using $a_k$, $b_k$, and $y_k$.
    *   Perform element-wise multiplication of the likelihood array and the current posterior array to compute the unnormalized posterior.
    *   **Sequential Normalization:** Calculate the area under the unnormalized curve by summing all elements and multiplying by $\Delta\theta$. Divide the unnormalized posterior array by this area to ensure it integrates to 1. 
4.  **Extract Estimates:** Find the mean (sum of $\theta \times \text{posterior} \times \Delta\theta$) and MAP (the $\theta$ value where the posterior array is at its maximum).

## 7. Evaluating Convergence over the Timeline


In [2]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# 1. Setup Simulation Parameters
np.random.seed(42)
n_items = 20
theta_true = 0.75

# Generate item parameters
b_params = np.random.normal(0, 1, n_items)
a_params = np.random.uniform(0.5, 2.0, n_items)

# Setup numerical grid for theta
grid_size = 1000
theta_grid = np.linspace(-4, 4, grid_size)
delta_theta = theta_grid[1] - theta_grid[0]

# 2PL probability function
def prob_correct(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

# 2. Tracking Variables
posterior = norm.pdf(theta_grid) # Initial prior: N(0,1)
map_estimates = [0.0]            # Prior MAP
mean_estimates = [0.0]           # Prior Mean

# 3. Simulate and Update
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]
    
    # Simulate user response
    p_true = prob_correct(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0
    
    # Calculate Likelihood for the grid
    p_grid = prob_correct(theta_grid, a_k, b_k)
    likelihood = (p_grid ** y_k) * ((1 - p_grid) ** (1 - y_k))
    
    # Update and Normalize Posterior
    unnormalized_posterior = posterior * likelihood
    normalization_constant = np.sum(unnormalized_posterior) * delta_theta
    posterior = unnormalized_posterior / normalization_constant
    
    # Extract MAP and Mean
    map_est = theta_grid[np.argmax(posterior)]
    mean_est = np.sum(theta_grid * posterior * delta_theta)
    
    map_estimates.append(map_est)
    mean_estimates.append(mean_est)

# 4. Visualize with Plotly
steps = list(range(n_items + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines+markers', name="Posterior Mean"))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name="MAP Estimate"))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*(n_items + 1), mode='lines', 
                         name="True Theta (0.75)", line=dict(dash='dash', color='red')))

fig.update_layout(title="Convergence of Sequential Bayesian Estimates",
                  xaxis_title="Item Number (k)",
                  yaxis_title="Latent Ability Estimate",
                  template="plotly_white")
fig.show()

### Analysis of Convergence

As $k$ increases, the distance between both estimators (Posterior Mean and MAP) and $\theta_{\text{true}}$ decreases, despite minor fluctuations caused by the randomness of individual responses. 

Each observation contributes new likelihood information that multiplies against the prior. As items accumulate, the product of these likelihoods overrides the standard normal prior. The running posterior distribution narrows and centers increasingly closer to the true parameter of 0.75. This behavior mathematically reflects the platform's growing measurement confidence, transitioning from an uncertain prior to a highly constrained, data-driven posterior.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

## 1. Structural Probability and Properties

The Beta distribution is defined by two shape parameters, $\alpha$ and $\beta$, which act as "pseudo-counts" of successes and failures.

*   **Uninformative state $(\alpha=1, \beta=1)$:** The density is perfectly flat (a Uniform distribution). The center of mass is exactly at 0.5, representing complete uncertainty where all values of $\theta$ are equally likely.
*   **Right-skewed state $(\alpha=2, \beta=8)$:** The parameter $\beta$ is dominant, pulling the probability mass strongly toward 0. The distribution has a long tail to the right (right-skewed), indicating a high belief that the true probability is low.
*   **Left-skewed state $(\alpha=8, \beta=2)$:** The parameter $\alpha$ is dominant, pulling the probability mass strongly toward 1. The distribution has a long tail to the left (left-skewed), indicating a high belief that the true probability is high.


In [3]:

import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta = np.linspace(0, 1, 500)

fig = go.Figure()

# (1, 1) Uninformative
fig.add_trace(go.Scatter(x=theta, y=beta.pdf(theta, 1, 1), 
                         name="alpha=1, beta=1 (Uniform)", line=dict(width=3)))

# (2, 8) Right-skewed
fig.add_trace(go.Scatter(x=theta, y=beta.pdf(theta, 2, 8), 
                         name="alpha=2, beta=8 (Right-skewed)", line=dict(width=3)))

# (8, 2) Left-skewed
fig.add_trace(go.Scatter(x=theta, y=beta.pdf(theta, 8, 2), 
                         name="alpha=8, beta=2 (Left-skewed)", line=dict(width=3)))

fig.update_layout(title="Beta Distribution Probability Density Functions",
                  xaxis_title="Conversion Rate (theta)",
                  yaxis_title="Density",
                  template="plotly_white")
fig.show()



## 2. Sequential Likelihood and Joint History

Because each impression is an independent Bernoulli trial, the likelihood of a single new response $y_k$ given $\theta$ is:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

The joint likelihood for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods from step 1 to step $k$:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} \theta^{y_j} (1 - \theta)^{1 - y_j} = \theta^{\sum_{j=1}^k y_j} (1 - \theta)^{k - \sum_{j=1}^k y_j}$$

## 3. Closed-Form Analytical Updates (Conjugacy)

Using Bayes' Theorem for a sequential update, the posterior at step $k$ is proportional to the product of the likelihood of $y_k$ and the prior (which is the posterior from step $k-1$):

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting the known forms:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right] \times \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right]$$

By combining the exponents, we get:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This resulting expression matches the exact functional form of a Beta distribution, proving that the Beta prior and Bernoulli likelihood are conjugate. The explicit closed-form parameter updates are simply:

$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

The Posterior Mean of the latent parameter $\Theta$ at time step $k$ evaluates directly to:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

## 4. Dynamic Shifting Mechanics

Because the updates are simple additions to the shape parameters:
*   An observed **click** ($y_k = 1$) increments $\alpha_k$ by 1, leaving $\beta_k$ unchanged. This mathematically shifts the distribution's center of mass to the right, toward a higher estimated conversion rate.
*   An observed **non-click** ($y_k = 0$) increments $\beta_k$ by 1, leaving $\alpha_k$ unchanged. This shifts the distribution's center of mass to the left, toward a lower estimated conversion rate.

**Contrast with Non-Conjugate Setups (e.g., 2PL IRT):** 
In this conjugate setup, updating our belief takes constant time via simple arithmetic on two parameters ($\alpha$ and $\beta$), maintaining an exact continuous functional form. In non-conjugate models like the 2PL IRT model, the posterior cannot be simplified algebraically. The platform must approximate the continuous distribution over a discrete grid, calculate the likelihood for every grid point, multiply arrays, and numerically integrate to compute a normalizing constant. This makes non-conjugate models significantly more computationally expensive to update per impression.

## 5. Running Point Estimators

Based on the updated parameters $\alpha_k$ and $\beta_k$ at step $k$, the exact closed-form point estimates are:

*   **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$):
    $$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$

*   **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$):
    $$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{defined for } \alpha_k, \beta_k \ge 1, \text{ excluding } \alpha_k=\beta_k=1)$$

## 6. Performance Tracking and Convergence Analysis

As $k$ approaches $100$, both the Posterior Mean and the MAP estimate converge strongly toward the true underlying rate $\theta_{\text{true}} = 0.35$. In the earliest steps, the estimates oscillate wildly due to small sample sizes. However, as evidence accumulates (higher $k$), the denominator of the estimators ($\alpha_k + \beta_k$) grows larger, making each new single impression have a mathematically smaller marginal impact on the overall fraction. 

This implies that while the uninformative initial prior ($\alpha_0=1, \beta_0=1$) dictates the starting point, the likelihood of the observed data rapidly overwhelms it. The influence of the prior decays at a rate of $\frac{1}{k}$, allowing the objective data to dictate the system's belief as time progresses.

In [4]:
import numpy as np
import plotly.graph_objects as go

# 1. Setup Simulation Parameters
np.random.seed(101)
n_impressions = 100
theta_true = 0.35

# 2. Tracking Variables
alpha_k = 1
beta_k = 1
mean_estimates = [alpha_k / (alpha_k + beta_k)]
map_estimates = [np.nan] # MAP is undefined at (1,1)

# 3. Simulate and Update
for k in range(n_impressions):
    # Simulate user interaction
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    
    # Analytical Update (Conjugacy)
    alpha_k += y_k
    beta_k += (1 - y_k)
    
    # Calculate Estimators
    mean_est = alpha_k / (alpha_k + beta_k)
    
    # Handle MAP edge cases
    if (alpha_k + beta_k - 2) == 0:
        map_est = np.nan
    else:
        # Clamp to [0, 1] for edge cases early in the sequence
        map_est = max(0.0, min(1.0, (alpha_k - 1) / (alpha_k + beta_k - 2)))
        
    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# 4. Visualize with Plotly
steps = list(range(n_impressions + 1))

fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines', name="Posterior Mean", line=dict(width=2)))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines', name="MAP Estimate", line=dict(width=2, dash='dot')))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*(n_impressions + 1), mode='lines', 
                         name="True Theta (0.35)", line=dict(dash='dash', color='red', width=2)))

fig.update_layout(title="Sequential CTR Estimation Convergence (Beta-Bernoulli)",
                  xaxis_title="Impression Number (k)",
                  yaxis_title="Estimated CTR (theta)",
                  template="plotly_white")
fig.show()

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## 1. Prior Belief Boundaries

The expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ for a Beta distribution is given by the formula $\frac{\alpha}{\alpha + \beta}$:

$$\mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$$

This distribution is highly appropriate for an initial prior because it assigns the vast majority of the probability mass near $1.0$ (representing an undamaged or lightly degraded state), which aligns with the physical reality of deploying a healthy component. The density gently tapers off toward the lower values, leaving a small non-zero probability for pre-existing manufacturing defects or undocumented damage without letting those scenarios dominate the baseline assumption.

In [5]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

theta_grid = np.linspace(0.01, 1.0, 500)
prior_density = beta.pdf(theta_grid, 8, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_grid, y=prior_density, mode='lines', name='Beta(8, 1.5) Prior'))
fig.update_layout(title='Initial Prior Belief of Stiffness Efficiency',
                  xaxis_title='Stiffness Efficiency (theta)',
                  yaxis_title='Density',
                  template='plotly_white')
fig.show()



## 2. Structural Likelihood Formulation

Taking the natural logarithm of the physical measurement model $y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}$ gives:

$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$

Since $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$, the variable $\ln(y_k)$ is normally distributed with mean $\mu = \ln(\theta K_{\text{nominal}})$ and variance $\sigma^2$. Therefore, $y_k$ follows a log-normal distribution. The likelihood contribution of a single measurement $y_k$ is:

$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{[\ln(y_k) - \ln(\theta K_{\text{nominal}})]^2}{2\sigma^2} \right)$$

Assuming the sensor measurements are conditionally independent given $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$ is the product of the individual likelihoods:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^{k} \frac{1}{y_j \sigma \sqrt{2\pi}} \exp\left( -\frac{[\ln(y_j) - \ln(\theta K_{\text{nominal}})]^2}{2\sigma^2} \right)$$

## 3. Mathematical Formulation of the Non-Conjugate Grid Update

An exact closed-form analytical solution does not exist because the Beta prior (characterized by a polynomial form $\theta^{\alpha-1}(1-\theta)^{\beta-1}$) and the log-normal likelihood (characterized by an exponential term $\exp(-(\ln \theta + C)^2)$) are not conjugate. When multiplied, they do not algebraically simplify into a recognizable, standard probability distribution family.

Instead, the recursive relationship must be evaluated numerically up to a proportionality constant:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \times f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

## 4. Running Point Estimates

Since we cannot express the posterior function analytically, the point estimators must be defined using formal integration and maximization over the bounded domain $(0, 1]$:

*   **Running Posterior Mean:**
    $$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$

*   **Running Maximum A Posteriori (MAP):**
    $$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \underset{\theta \in (0, 1]}{\operatorname{arg\,max}} \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

## 5. Algorithmic Grid Approximation and Normalization

To handle this non-conjugate update computationally:

1.  **Grid Initialization:** Define a discrete, finely spaced one-dimensional array corresponding to $\theta$ from a small value $\epsilon$ (e.g., 0.01) to 1.0. We strictly avoid $0.0$ to prevent undefined $\ln(0)$ evaluations in the likelihood function.
2.  **Prior Array:** Evaluate the initial Beta(8, 1.5) density at each point on the grid to create the starting posterior array.
3.  **Sequential Update:** For each new measurement $y_k$:
    *   Evaluate the log-normal likelihood formula $L(y_k \mid \theta)$ across the entire grid.
    *   Perform element-wise multiplication of the likelihood array and the current posterior array to yield the unnormalized posterior array.
    *   **Normalization:** Compute the definite integral of the unnormalized array using the trapezoidal rule. Divide the unnormalized posterior array by this scalar area value so the density integrates precisely to 1.0.
4.  **Extract Estimates:** Compute the Posterior Mean by numerically integrating $\theta \cdot f(\theta)$ via the trapezoidal rule, and the MAP by identifying the $\theta$ value at the maximum array index.

## 6. Performance Tracking and Degradation Convergence Analysis

### Analysis of the System

The progression shows that it takes approximately 3 to 5 sensor readings for the likelihood functions to overpower the heavily skewed, optimistic initial prior. By step 5, the density has fundamentally shifted away from 0.84 and centered near the true state of 0.68.

As $k$ approaches 15, the posterior density curves become progressively narrower (reduced variance) and taller (higher peak density). This narrowing implies that the system is growing increasingly confident in its estimation of the damage state. In an engineering context, this sharp certainty is critical—tight confidence intervals prevent false-positive alarms while ensuring that safety threshold decisions (e.g., grounding an aircraft if the 95% credible interval drops below $\theta=0.70$) are driven by hard, statistical evidence rather than baseline assumptions.

In [6]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
from plotly.subplots import make_subplots

# 1. Simulation Parameters
np.random.seed(42)
n_steps = 15
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15

# Generate noisy sensor readings
epsilon = np.random.normal(0, sigma, n_steps)
y_measurements = theta_true * K_nominal * np.exp(epsilon)

# 2. Grid Setup
grid_points = 1000
theta_grid = np.linspace(0.01, 1.0, grid_points)

# Trackers
posterior = beta.pdf(theta_grid, 8, 1.5)
posteriors_to_plot = {0: posterior.copy()}
mean_estimates = [np.trapezoid(theta_grid * posterior, x=theta_grid)]
map_estimates = [theta_grid[np.argmax(posterior)]]

# 3. Sequential Update Loop
for k, y_k in enumerate(y_measurements, start=1):
    # Calculate Likelihood
    likelihood = (1 / (y_k * sigma * np.sqrt(2 * np.pi))) * \
                 np.exp(- (np.log(y_k) - np.log(theta_grid * K_nominal))**2 / (2 * sigma**2))
    
    # Update and Normalize
    unnormalized = posterior * likelihood
    normalization_constant = np.trapezoid(unnormalized, x=theta_grid)
    posterior = unnormalized / normalization_constant
    
    # Store selected posteriors for visualization
    if k in [1, 2, 5, 10, 15]:
        posteriors_to_plot[k] = posterior.copy()
        
    # Point Estimates
    mean_est = np.trapezoid(theta_grid * posterior, x=theta_grid)
    map_est = theta_grid[np.argmax(posterior)]
    
    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# 4. Visualization
fig = make_subplots(rows=1, cols=2, subplot_titles=("Evolution of Posterior Density", "Convergence of Estimators"),
                    horizontal_spacing=0.15)

# Plot 1: Posterior densities
colors = ['#c8d6e5', '#8395a7', '#576574', '#222f3e', '#10ac84', '#ee5253']
for idx, (k, post_density) in enumerate(posteriors_to_plot.items()):
    fig.add_trace(go.Scatter(x=theta_grid, y=post_density, mode='lines', 
                             name=f'Step {k}', line=dict(color=colors[idx], width=2)), row=1, col=1)

# Plot 2: Convergence
steps = list(range(n_steps + 1))
fig.add_trace(go.Scatter(x=steps, y=mean_estimates, mode='lines+markers', name='Posterior Mean'), row=1, col=2)
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines+markers', name='MAP Estimate'), row=1, col=2)
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*(n_steps+1), mode='lines', 
                         name='True Theta (0.68)', line=dict(dash='dash', color='red')), row=1, col=2)

fig.update_layout(template='plotly_white', height=500, title_text="Structural Health Monitoring: Bayesian Updating")
fig.update_xaxes(title_text="Stiffness Efficiency (theta)", row=1, col=1)
fig.update_yaxes(title_text="Density", row=1, col=1)
fig.update_xaxes(title_text="Inspection Step (k)", row=1, col=2)
fig.update_yaxes(title_text="Estimated Efficiency", row=1, col=2)
fig.show()

# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

## 1. Deriving the Marginal Density

By the law of total probability, the marginal distribution of $X_i$ is found by marginalizing out the discrete latent variable $C_i$ over its $K$ possible states:

$$p(x_i)=\sum_{k=1}^K P(C_i=k)p(X_i=x_i\mid C_i=k)$$

Substituting the given prior $P(C_i=k)=\phi_k$ and the conditional Gaussian density $X_i\mid C_i=k \sim \mathscr{N}(\mu_k,\Sigma_k)$, we get:

$$p(x_i)=\sum_{k=1}^K \phi_k \mathscr{N}(x_i\mid \mu_k,\Sigma_k)$$

This density is called a **Gaussian mixture density** because the overall probability distribution of the data is a convex combination (or "mixture") of several individual Gaussian components. The parameters $\phi_k$ act as mixing weights that sum to 1, ensuring the resulting combination is a valid probability density function.

---

## 2. Deriving the Posterior Cluster Probability

Using Bayes' rule, the probability of the latent cluster assignment $C_i$ given the observed data $x_i$ is proportional to the likelihood of the data given the cluster multiplied by the prior probability of the cluster:

$$P(C_i=k\mid X_i=x_i)=\frac{P(X_i=x_i\mid C_i=k)P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)}$$

Substituting the Gaussian model and the cluster prior yields the responsibility:

$$P(C_i=k\mid X_i=x_i)=\frac{\phi_k\mathscr{N}(x_i\mid \mu_k,\Sigma_k)}{\sum_{j=1}^K \phi_j\mathscr{N}(x_i\mid \mu_j,\Sigma_j)}$$

The quantity $\gamma_{ik}$ is interpreted as a **posterior probability** because it represents our updated belief about which cluster generated $x_i$ *after* observing the data. We start with a prior belief $\phi_k$, evaluate how well the cluster $k$ explains the specific observation $x_i$ via the likelihood $\mathscr{N}(x_i\mid \mu_k,\Sigma_k)$, and normalize this across all possible clusters.

---

## 3. One-Hot Encoding of the Latent Cluster Variable

Since $Z_{ik}$ is an indicator (Bernoulli) random variable taking values in $\{0, 1\}$, its expectation is simply the probability that it equals $1$:

$$\mathbb{E}[Z_{ik}\mid X_i=x_i] = 1 \cdot P(Z_{ik}=1\mid X_i=x_i) + 0 \cdot P(Z_{ik}=0\mid X_i=x_i)$$

Because $Z_{ik}=1$ if and only if $C_i=k$, this simplifies directly to:

$$\mathbb{E}[Z_{ik}\mid X_i=x_i] = P(C_i=k\mid X_i=x_i) = \gamma_{ik}$$

By assembling these expected values into the vector $Z_i$, we apply the expectation operator element-wise:

$$\mathbb{E}[Z_i\mid X_i=x_i]=\begin{bmatrix}\mathbb{E}[Z_{i1}\mid X_i=x_i]\\ \mathbb{E}[Z_{i2}\mid X_i=x_i]\\ \vdots\\ \mathbb{E}[Z_{iK}\mid X_i=x_i]\end{bmatrix}=\begin{bmatrix}\gamma_{i1}\\ \gamma_{i2}\\ \vdots\\ \gamma_{iK}\end{bmatrix}$$

This confirms that the soft cluster assignment in a GMM is mathematically identical to the conditional expectation of the latent state vector given the data.

---

## 4. From Soft Assignment to Hard Clustering

*   **Soft clustering** provides a probability distribution over all available clusters (the vector $\mathbb{E}[Z_i\mid X_i=x_i]$). It acknowledges uncertainty; a point situated between two cluster centers might have a soft assignment of $[0.5, 0.5]$.
*   **Hard clustering** forces a discrete, deterministic decision. By applying the $\operatorname{arg\,max}$ operator, we collapse the probability vector into a single integer label $\widehat{C}_i$, stripping away the confidence intervals and boundary ambiguity.

---

## 5. Conditional Expectation of the Observation Given the Cluster

By the definition of the generative model, conditional on $C_i=k$, the data is drawn from $\mathscr{N}(\mu_k,\Sigma_k)$. The expected value of a Gaussian distribution is its mean vector:

$$\mathbb{E}[X_i\mid C_i=k]=\mu_k$$

Here, $\mu_k$ acts as the **center** of cluster $k$ in the $d$-dimensional feature space—it is the average location of all points generated by this specific component.

**Comparing the two conditional expectations:**
*   $\mathbb{E}[Z_i\mid X_i=x_i]$ operates on the latent space. It maps a fixed physical location $x_i$ back to a probability distribution over the cluster identities (soft membership).
*   $\mathbb{E}[X_i\mid C_i=k]$ operates on the data space. It takes a fixed cluster identity $k$ and projects it into the physical feature space to find its expected coordinate location (the mean).

---

## 6. The Complete-Data Likelihood

The complete-data likelihood is:

$$p(x,z)=\prod_{i=1}^n \prod_{k=1}^K \left[\phi_k\mathscr{N}(x_i\mid \mu_k,\Sigma_k)\right]^{z_{ik}}$$

Taking the natural logarithm, the product over $n$ and $K$ becomes a double summation, and the exponent $z_{ik}$ drops down:

$$\ell_c=\sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[\log \phi_k + \log \mathscr{N}(x_i\mid \mu_k,\Sigma_k)\right]$$

If the $z_{ik}$ values were known (hard integers), maximizing this would be trivial. The indicator $z_{ik}$ zeroes out the terms for all clusters except the true one. The parameters for each cluster $k$ could be estimated completely independently of the others using standard Maximum Likelihood Estimation (MLE) on only the subset of data points assigned to cluster $k$.

---

## 7. The EM Interpretation

Because we don't know the true labels $z_{ik}$, the Expectation step (E-step) of the EM algorithm estimates them using the current model parameters. We replace the missing indicator variables with their conditional expectations $\gamma_{ik}$:

$$Q=\sum_{i=1}^n \sum_{k=1}^K \gamma_{ik}\left[\log \phi_k + \log \mathscr{N}(x_i\mid \mu_k,\Sigma_k)\right]$$

The E-step is a **conditional update of cluster membership probabilities**. Given our current best guess of where the clusters are ($\mu_k, \Sigma_k$) and their sizes ($\phi_k$), we re-evaluate the posterior probability $\gamma_{ik}$ for every point. We construct a proxy function $Q$ that weights the log-likelihood of each data point across all clusters according to these soft responsibilities.

---

## 8. Parameter Updates

In the Maximization step (M-step), we maximize the $Q$ function with respect to the parameters. The resulting updates are:

*   $N_k=\sum_{i=1}^n \gamma_{ik}$
*   $\phi_k^{\text{new}}=\frac{N_k}{n}$
*   $\mu_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}x_i$
*   $\Sigma_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^T$

The responsibility $\gamma_{ik}$ acts as a **fractional membership weight**. Instead of a data point $x_i$ entirely belonging to one cluster and shifting its mean $\mu_k$ at full force, the point pulls on all $K$ cluster centers simultaneously, weighted proportionally by $\gamma_{ik}$.

---

## 9. Interpretation

Gaussian Mixture Model clustering is fundamentally a repeated process of conditional updating rooted in probabilistic principles. The process begins with the mixture weight $\phi_k$, which serves as the prior probability of cluster $k$ before any data is evaluated. Upon observing a data point $x_i$, the Gaussian density $\mathscr{N}(x_i\mid \mu_k,\Sigma_k)$ acts as a likelihood function, measuring exactly how statistically compatible $x_i$ is with cluster $k$. Bayes' rule merges these two components to compute the responsibility $\gamma_{ik}$, representing the updated posterior probability of cluster $k$. This responsibility forms the soft assignment vector $\mathbb{E}[Z_i\mid X_i=x_i]$. During the M-step, these conditional expectations dictate how the model adapts; rather than rigidly assigning points, the algorithm updates the cluster parameters using these posterior membership probabilities as fractional weights. Consequently, GMM clustering is not merely drawing boundaries, but continuously refining a probabilistic representation of latent cluster memberships via conditional expectations.

---

## 10. Computational Simulation and Out-of-Sample Validation



### Evaluation of the Visualizations and the Expectation Vector

The continuous background contour map visually demonstrates the mathematical properties proven in Part 3. By calculating the responsibility over a dense grid of coordinates $x_{\text{grid}}$, we calculate the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ at every location. 

The color mapping plots the maximum of this vector (the maximum responsibility $\max_{k} \gamma_{ik}$). In regions near the structural cluster centers ($\mu_k$), the color indicates high confidence (approaching 1.0), meaning the expectation vector is highly localized to a single one-hot encoded state. As the physical coordinate traverses the space and transitions between two clusters, the contour visually darkens or lightens in the valleys. These valleys represent the decision boundaries where $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ becomes uniformly distributed (e.g., $[0.5, 0.5]$), vividly mapping the inherent mathematical uncertainty of probabilistic clustering onto the feature space. Overlaying out-of-sample test points confirms that the regions of highest latent ambiguity are consistent between both seen and unseen data.

In [14]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class GMMFinancialSegmenter:
    def __init__(self, data_url, features=['PURCHASES', 'CREDIT_LIMIT'], n_components=3, random_state=42):
        self.data_url = data_url
        self.features = features
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(
            n_components=self.n_components, 
            covariance_type='full', 
            random_state=self.random_state
        )
        
    def load_and_preprocess(self):
        print("Downloading dataset directly from the web...")
        # Read the raw CSV directly from a public GitHub repository hosting the Kaggle data
        df = pd.read_csv(self.data_url)
        df = df.dropna(subset=self.features)
        
        X = df[self.features].values
        X_scaled = self.scaler.fit_transform(X)
        
        self.X_train, self.X_test = train_test_split(
            X_scaled, test_size=0.2, random_state=self.random_state
        )
        print(f"Data successfully loaded and split: {len(self.X_train)} train samples, {len(self.X_test)} test samples.")
        
    def fit_evaluate(self):
        print("Fitting Gaussian Mixture Model...")
        # EM Execution
        self.gmm.fit(self.X_train)
        
        print(f"EM Algorithm Converged: {self.gmm.converged_}")
        print(f"Number of Iterations: {self.gmm.n_iter_}")
        
        # Out-of-Sample Validation
        test_log_likelihood = self.gmm.score(self.X_test)
        print(f"Average Out-of-Sample Log-Likelihood: {test_log_likelihood:.4f}")
        
    def _create_mesh_grid(self, padding=1.0, resolution=150):
        # Create a coordinate grid spanning the data range
        x_min, x_max = self.X_train[:, 0].min() - padding, self.X_train[:, 0].max() + padding
        y_min, y_max = self.X_train[:, 1].min() - padding, self.X_train[:, 1].max() + padding
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution), 
                             np.linspace(y_min, y_max, resolution))
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        return xx, yy, grid_points

    def generate_visualizations(self):
        print("Generating visualizations...")
        xx, yy, grid_points = self._create_mesh_grid()
        
        # Calculate responsibilities (gamma_ik) for the grid points
        # predict_proba returns E[Z_i | X_i = x_grid]
        grid_responsibilities = self.gmm.predict_proba(grid_points)
        max_responsibilities = grid_responsibilities.max(axis=1).reshape(xx.shape)

        fig = make_subplots(rows=1, cols=3, 
                            subplot_titles=('Raw Training Density', 
                                            'Training Assignments (Soft Contour)', 
                                            'Test Assignments on Contour'),
                            horizontal_spacing=0.05)

        # 1. 2D Density Heatmap
        fig.add_trace(go.Histogram2dContour(
            x=self.X_train[:, 0], y=self.X_train[:, 1],
            colorscale='Blues', reversescale=False,
            xaxis='x1', yaxis='y1'
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=self.X_train[:, 0], y=self.X_train[:, 1],
            mode='markers', marker=dict(size=2, color='black', opacity=0.3)
        ), row=1, col=1)

        # 2. Training Assignments with Contours
        fig.add_trace(go.Contour(
            x=xx[0], y=yy[:,0], z=max_responsibilities,
            colorscale='Viridis', opacity=0.5, showscale=False
        ), row=1, col=2)
        train_labels = self.gmm.predict(self.X_train)
        fig.add_trace(go.Scatter(
            x=self.X_train[:, 0], y=self.X_train[:, 1], mode='markers',
            marker=dict(size=3, color=train_labels, colorscale='Plasma', opacity=0.7)
        ), row=1, col=2)

        # 3. Test Assignments on Contour
        fig.add_trace(go.Contour(
            x=xx[0], y=yy[:,0], z=max_responsibilities,
            colorscale='Viridis', opacity=0.5, showscale=True,
            colorbar=dict(title='Max Responsibility', x=1.02)
        ), row=1, col=3)
        test_labels = self.gmm.predict(self.X_test)
        fig.add_trace(go.Scatter(
            x=self.X_test[:, 0], y=self.X_test[:, 1], mode='markers',
            marker=dict(size=4, color=test_labels, colorscale='Plasma', opacity=0.9,
                        line=dict(width=0.5, color='white'))
        ), row=1, col=3)

        fig.update_layout(
            height=500, width=1400, 
            title_text="GMM Segmentation: From Data Density to Probabilistic Clusters",
            showlegend=False,
            template="plotly_white"
        )
        
        for i in range(1, 4):
            fig.update_xaxes(title_text=self.features[0] + " (Scaled)", row=1, col=i)
            fig.update_yaxes(title_text=self.features[1] + " (Scaled)", row=1, col=i)

        fig.show()


if __name__ == "__main__":
    # Public Raw GitHub URL containing the specific CC GENERAL.csv dataset
    RAW_DATA_URL = "https://raw.githubusercontent.com/RAHULKASHYAP02/Credit-Card-Segmentation/master/CC%20GENERAL.csv"
    
    # Initialize, run, and plot
    segmenter = GMMFinancialSegmenter(data_url=RAW_DATA_URL)
    segmenter.load_and_preprocess()
    segmenter.fit_evaluate()
    segmenter.generate_visualizations()

Data successfully loaded and split: 7159 train samples, 1790 test samples.
Fitting Gaussian Mixture Model...
EM Algorithm Converged: True
Number of Iterations: 19
Average Out-of-Sample Log-Likelihood: -1.6465
Generating visualizations...
